# M07b.3/.4: final-eval no-regression, re-probe, then freeze D1

**Thin launcher.** The logic, locks and gates live in the repo:

    python -m pii_redteam.cli freeze-d1 --config configs/m07b_freeze_d1_v2.yaml \
        --source-run-dir <Drive>/pii-redteam-rl/d1v2-<timestamp>

**Runtime -> Change runtime type -> T4 GPU, then Run all.** `VERSION` in the setup cell
picks the training run: `"v2"` (default) freezes the newest `d1v2-*` run (epochs 2 and 3
saved); `"v1"` freezes `d1-20261108-145253` (one candidate). It needs frozen D0 at
`pii-redteam-rl/d0-20261004-142714/d0-best` and the dataset cache on Drive. The probe also
downloads Qwen/Qwen3.5-2B. For train + freeze in one unattended run, use
`M07b_d1_overnight.ipynb`.

In order, stopping at the first failed lock:

1. **Locks and the choice:** final-eval partition sha, D0 sha, candidate shas (vs the
   training report, and config pins when set), training and M07a configs unchanged, repo
   commits recorded, no existing frozen directory. With several saved epochs, the
   candidate that may be frozen is fixed **here** by the pre-registered dev-only rule in
   the training config; nothing scored later can change it.
2. **Final-eval (single use per candidate):** every candidate vs D0 on the held-out
   AI4Privacy partition, side by side. Gates on the chosen candidate: span-F1 drop
   <= 0.01, the pre-registered precision/recall drop <= 0.01, and the precision guard
   (name-free flagged-row rate at most +30% relative to D0's; owner decision 2026-11-09).
   An existing per-candidate final-eval report is reused, never re-scored.
3. **Short negatives:** `data/synthetic/short_eval_neg_v1.jsonl` (sha pinned). Gated on the
   95 place / org_product / calendar / role_only rows (proposed: <= 5% flagged); the 16
   homograph rows are reported D0 vs each candidate only.
4. **M07a probe:** the unchanged M07a probe with the chosen candidate as the scorer; D0 is
   rescored on the same texts. Must pass the go/no-go table with the miss rate inside the
   1-30% band. Other candidates are probed only if `probe.probe_all_candidates` is true.
5. **Freeze:** only if every required check passes, the chosen candidate is copied to
   `pii-redteam-rl/d1-frozen-<run_id>/` with `freeze_manifest.json`. Otherwise the status
   is `freeze_blocked` with the failing checks. A frozen directory is never overwritten.

A blocked freeze is a recorded result, not a reason to edit a threshold.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/GauthierRoy/pii-redteam-rl.git"
REPO_DIR = "/content/pii-redteam-rl"
DRIVE_ROOT = "/content/drive/MyDrive/pii-redteam-rl"
# Change to "main" once feat/m07b-freeze-d1 is merged.
REF = "feat/m07b-freeze-d1"

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Always sync to the target ref: a reused VM may hold a STALE clone.
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "fetch", "-q", "origin", REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "-q", "-B", REF, f"origin/{REF}"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{REF}"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", REF, REPO_URL, REPO_DIR], check=True)
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
print("repo commit:", commit)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate"], check=True
)

import torch

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU."
free_gb = torch.cuda.mem_get_info()[0] / 1e9
assert free_gb >= 10, (
    f"GPU has only {free_gb:.1f} GB free - dirty runtime. Runtime -> Disconnect and"
    " delete runtime, then reopen from GitHub and Run all."
)
print(f"GPU free: {free_gb:.1f} GB - OK ({torch.cuda.get_device_name(0)})")
ENV = dict(os.environ, PYTHONPATH=os.path.join(REPO_DIR, "src"), PROBE_REPO_COMMIT=commit)

VERSION = "v2"  # "v1" = d1-20261108-145253 (one candidate)
if VERSION == "v2":
    import glob

    CONFIG = "configs/m07b_freeze_d1_v2.yaml"
    runs = sorted(glob.glob(os.path.join(DRIVE_ROOT, "d1v2-2*")))
    assert runs, "No d1v2-* training run on Drive: run M07b_train_d1.ipynb (v2) first."
    SOURCE_DIR = runs[-1]
    EXTRA = ["--source-run-dir", SOURCE_DIR]
else:
    CONFIG = "configs/m07b_freeze_d1.yaml"
    SOURCE_DIR = os.path.join(DRIVE_ROOT, "d1-20261108-145253")
    EXTRA = []
print("config:", CONFIG, "| source run:", SOURCE_DIR)

In [ ]:
import glob
import json

# Read-only preflight: what is about to be scored. Nothing is scored here.
with open(os.path.join(SOURCE_DIR, "d1_report.json"), encoding="utf-8") as f:
    source = json.load(f)
print("source run:", source["run_id"], "| status:", source["status"])
print("training repo commit:", source.get("environment", {}).get("repo_commit"))
for key, ckpt in (source.get("epoch_checkpoints") or {}).items():
    print(f"  epoch {key}: {ckpt['dir']} sha {ckpt['sha256'][:16]} dev F1 {ckpt['dev_span_f1']}"
          f" name-free dev rows flagged {ckpt['dev_neg_rows_with_pred']}")
print("freeze rule choice:", (source.get("freeze_selection") or {}).get("chosen_epoch"),
      "(v1 run: the single d1-candidate)")
print("final-eval reports already present (reused, never re-scored):",
      [os.path.basename(p) for p in glob.glob(os.path.join(SOURCE_DIR, "d1_final_eval_report*"))])
frozen = os.path.join(DRIVE_ROOT, "d1-frozen-" + source["run_id"])
print("frozen dir exists:", os.path.exists(frozen), "(the run refuses to overwrite it)")
neg = os.path.join(REPO_DIR, "data/synthetic/short_eval_neg_v1.jsonl")
print("short negatives file present:", os.path.exists(neg), "(pinned: missing blocks the freeze)")

## Run the freeze

Long step: final-eval, then the probe's timing pilot and generation.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pii_redteam.cli", "freeze-d1", "--config", CONFIG, *EXTRA],
    cwd=REPO_DIR,
    env=ENV,
)
print(f"\nfreeze-d1 exit code: {result.returncode} (0 = frozen, 2 = freeze_blocked)")
if result.returncode not in (0, 2):
    print("^^ the traceback above is the real error - paste it back to me")
prefix = "d1v2-freeze" if VERSION == "v2" else "d1-freeze"
RUN_DIR = sorted(glob.glob(os.path.join(DRIVE_ROOT, f"{prefix}-2*")))[-1]
print("latest freeze run dir:", RUN_DIR)

In [ ]:
def print_freeze_summary(report_path):
    with open(report_path, encoding="utf-8") as f:
        report = json.load(f)
    print("STATUS:", report["status"])
    print("chosen candidate:", report.get("chosen_candidate"),
          "| rule:", (report.get("selection") or {}).get("rule"))
    for row in report.get("side_by_side") or []:
        mark = "CHOSEN" if row["chosen"] else "      "
        print(f"  {mark} {row['name']} (epoch {row['epoch']}): final-eval F1 "
              f"{row['final_eval_f1']} P {row['final_eval_precision']} R "
              f"{row['final_eval_recall']} name-free rows flagged "
              f"{row['final_eval_neg_rows_with_pred']} | short-neg gated rate "
              f"{row['short_negatives_gated_fp_row_rate']} | probe ok {row['probe_ok']}")
        print(f"         gates {row['gates_ok']}")
    print("checks:")
    for name, c in report["checks"].items():
        req = "" if c["required"] else " (optional)"
        print(f"  [{c['status'].upper()}] {name}{req}")
    if report["status"] == "frozen":
        print("frozen dir:", report.get("frozen_dir"))
    print("report:", report_path)
    print("\nexperiments_entry (paste into EXPERIMENTS.yaml):")
    print(json.dumps(report.get("experiments_entry"), indent=2))
    return report


print_freeze_summary(os.path.join(RUN_DIR, "d1_freeze_report.json"))

**When the run finishes:** bring back `d1_freeze_report.json` from the freeze run
directory (and the probe's `m07a-d1*-<timestamp>/probe_report.json`). The report's
`experiments_entry` goes into `EXPERIMENTS.yaml`.

**After a disconnect:** per-candidate final-eval reports are kept in the training run
directory and reused, not re-scored. With the v2 config the newest finished probe run for
the chosen sha is reused automatically (`probe.reuse_run_dir: auto`); a probe that did not
finish is regenerated. Run all again.

**If blocked:** the failing checks are listed above. Do not change a threshold or the epoch
choice to get a pass without a recorded owner decision.